# MIA Visualization

In [1]:
import os
from pathlib import Path
from typing import Any
import math

import ipywidgets as widgets
import plotly.express as px
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
steps_per_epoch = math.ceil(10000/4096)

In [3]:
train_steps_dcr = [0.506, 0.510, 0.520, 0.519, 0.516, 0.516, 0.518, 0.516]
train_steps_wb = [0.155, 0.297, 0.408, 0.435, 0.432, 0.449, 0.423, 0.461]
train_steps_bb = [0.108, 0.131, 0.138, 0.157, 0.153, 0.152, 0.161, 0.177]
train_steps = [epochs*steps_per_epoch for epochs in [1e03, 2e03, 3e03, 4e03, 5e03, 6e03, 7e03, 8e03]]

diffusion_steps_dcr = [0.504, 0.518, 0.518]
diffusion_steps_bb = [0.104, 0.162, 0.168]
diffusion_steps = [5, 50, 300]

synthetic_size = ["0.5x","1x", "2x"]
bb_10k = [0.113, 0.177, 0.225]
bb_20k = []
dcr_10k = [0.519, 0.518, 0.520]

batch_size_dcr = [0.524, 0.516, 0.510]
batch_size_wb = [0.745, 0.461, 0.186]
batch_size_bb = [0.191, 0.177, 0.121]
batch_size = [2048, 4096, 8192]

train_size = [5e03, 1e04, 2e04]
train_size_wb = [0.610, 0.461, 0.279]
train_size_bb = [0.220, 0.177, 0.098]
train_size_dcr = [0.512, 0.516, 0.522]
train_size_ideal_dcr = [0.5, 0.5, 0.5]

model_variation = ["512x1", "1024x1", "512x2", "1024x2", "512x3", "1024x3"]
mia_wb = [0.257, 0.407, 0.293, 0.461, 0.301, 0.452]
mia_bb = [0.114, 0.145, 0.127, 0.177, 0.123, 0.181]
dcr = [0.505, 0.510, 0.512, 0.516, 0.508, 0.539]
dcr_ideal = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
integers = list(range(len(model_variation)))

In [4]:
train_steps

[3000.0, 6000.0, 9000.0, 12000.0, 15000.0, 18000.0, 21000.0, 24000.0]

In [5]:
FIG_HEIGHT = 800
FIG_WIDTH = 1200
FONT_SIZE = 44

## Utilities

In [6]:
def format_metric_name(metric_name: str) -> str:
    """
    Prettify diagram texts by replacing
    snake case with regular title case.
    """
    output_words = []
    for word in metric_name.split("_"):
        word = word.title() if word not in ["FPR", "TPR"] else word.upper()
        output_words.append(word)

    return " ".join(output_words)

## Plotting

In [7]:
repo_abs_path = Path(os.path.abspath("")).parent.parent

plots_dir = f"{repo_abs_path}/examples/visualizations/diabetes_tf_training_tabdiff"

In [8]:
def customize_figure_layout(gen_fig: Any, width = FIG_WIDTH, height = FIG_HEIGHT) -> None:
    gen_fig.update_xaxes(automargin=True)
    gen_fig.update_yaxes(automargin=True)

    gen_fig.update_layout(
        height=height,
        width=width,
        font_color="black",
        legend=dict(
            y=1.0,
            x=0.5,
            xanchor="center",
            yanchor="bottom",
            orientation="h",
            valign="top",
            title_text="",
            font=dict(size=FONT_SIZE-4),
            title_font_family="Helvetica",
        ),
        title_font_family="Helvetica",
        title_x=0.5,
        title_y=0.99,
        margin=dict(l=0, r=0, t=40, b=0, pad=0),
        plot_bgcolor="white",
        font=dict(size=FONT_SIZE, family="Helvetica"),
    )

def trim_png_whitespace(image_path: str, pad: int = 2) -> None:
    """Crop near-white borders from a saved PNG."""
    import numpy as np
    from PIL import Image

    img = Image.open(image_path)
    arr = np.asarray(img)
    content = np.any(arr[:, :, :3] < 250, axis=2) if arr.ndim == 3 else arr < 250
    rows = np.any(content, axis=1)
    cols = np.any(content, axis=0)
    if not rows.any() or not cols.any():
        return

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    rmin = max(0, int(rmin) - pad)
    cmin = max(0, int(cmin) - pad)
    rmax = min(arr.shape[0] - 1, int(rmax) + pad)
    cmax = min(arr.shape[1] - 1, int(cmax) + pad)
    img.crop((cmin, rmin, cmax + 1, rmax + 1)).save(image_path)


def save_and_display_figure(gen_fig: Any, template_name: str, plots_dir: str) -> None:
    plot_png_path = f"{plots_dir}/{template_name}.png"
    plot_pdf_path = f"{plots_dir}/{template_name}.pdf"

    os.makedirs(plots_dir, exist_ok=True)
    gen_fig.write_image(plot_png_path, scale=2)
    gen_fig.write_image(plot_pdf_path)
    trim_png_whitespace(plot_png_path)

    display(gen_fig)

In [15]:
df = {"Train Steps": train_steps, "TF (WB)": train_steps_wb, "TF (BB)": train_steps_bb, "DCR": train_steps_dcr}

fig = px.line(
    data_frame=df,
    x="Train Steps",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,
)
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=0.5, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.data[0].line.color = "#53abff"
fig.data[1].line.color = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[6e03, 1.2e04, 1.8e04, 2.4e04],
    ),
)
fig.update_xaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".2E"
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "diabetes_tf_dcr_train_steps", plots_dir)

In [10]:
df = {"Diffusion Steps": diffusion_steps, "TF (BB)": diffusion_steps_bb, "DCR": diffusion_steps_dcr}

fig = px.line(
    data_frame=df,
    x="Diffusion Steps",
    y=["TF (BB)", "DCR"],
    markers=True,
)
fig.data[0].line.color = "#f78d8d"
fig.data[1].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=0.5, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[5e0, 5e1, 3e2],
    ),
    yaxis=dict(
        tickmode='array',
        tickvals=[0.1, 0.2, 0.3, 0.4, 0.5],
    ),
)
fig.update_xaxes(
    type="log", ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E"
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "diabetes_tf_dcr_diffusion_steps", plots_dir)

In [16]:
df = {"Synthetic Size": synthetic_size, "TF (BB) 10K": bb_10k, "DCR 10K": dcr_10k}

fig = px.line(
    data_frame=df,
    x="Synthetic Size",
    y=["TF (BB) 10K", "DCR 10K"],
    markers=True,
)
fig.data[0].line.color = "#f78d8d"
fig.data[1].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=0.5, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=["0.5x", "1x", "2x"],
    ),
)
fig.update_xaxes(
   ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "diabetes_tf_dcr_synthetic_size", plots_dir)

In [12]:
df = {"Batch Size": batch_size, "TF (WB)": batch_size_wb, "TF (BB)": batch_size_bb, "DCR": batch_size_dcr}

fig = px.line(
    data_frame=df,
    x="Batch Size",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,
    log_x=True,

)
fig.data[0].line.color = "#53abff"
fig.data[1].line.color = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=0.5, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=["2048", "4096", "8192"],
    ),
)
fig.update_xaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "diabetes_tf_dcr_batch_size", plots_dir)

In [13]:
df = {"Train Size": train_size, "TF (WB)": train_size_wb, "TF (BB)": train_size_bb, "DCR": train_size_dcr, "Ideal DCR": train_size_ideal_dcr}

fig = px.line(
    data_frame=df,
    x="Train Size",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,

)
fig.add_trace(go.Scatter(x=df["Train Size"], y=df["Ideal DCR"], mode='lines', name='Ideal DCR'))
fig.data[0].line.color = "#53abff"
fig.data[1].line.color = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.data[3].line.color = "#660066"
fig.data[3].line.dash = "dash"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[5e03, 1e04, 2e04],
    ),
)
fig.update_xaxes(
    type="log", ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE, tickformat=".1E",
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "diabetes_tf_dcr_train_size", plots_dir)

In [14]:
df = {"integers": integers, "TF (WB)": mia_wb, "TF (BB)": mia_bb, "DCR": dcr}

fig = px.line(
    data_frame=df,
    x="integers",
    y=["TF (WB)", "TF (BB)", "DCR"],
    markers=True,

)
fig.data[0].line.color = "#53abff"
fig.data[1].line.color  = "#f78d8d"
fig.data[2].line.color = "#660066"
fig.add_hline(y=0.1, line_dash="dash", line_color="red", annotation_text="Ideal MIA", annotation_position="bottom right", line_width=6)
fig.add_hline(y=1/2, line_dash="dash", line_color="#660066", annotation_text="Ideal DCR", annotation_position="bottom right", line_width=6)
fig.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=integers,
        ticktext=model_variation,
    ),
)
fig.update_xaxes(
   ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Model Variations", tickangle=-25, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_yaxes(
    ticks="outside", tickwidth=4, ticklen=7.5, linewidth=4, linecolor="black", title = "Metric Score", tickangle=0, title_font_family="Helvetica", tickfont_family="Helvetica", tickfont_size=FONT_SIZE,
)
fig.update_traces(marker=dict(size=20), line=dict(width=8))

customize_figure_layout(fig, width=1400, height=900)
fig.update_layout(yaxis_title=None)
save_and_display_figure(fig, "diabetes_tf_dcr_model_variation", plots_dir)